##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

DiffusionGemma 是一個實驗性的開放模型，探索文字擴散（text diffusion）——一種極為快速的文字生成方式。基於 26B（4B 啟動）混合專家（MoE）Gemma 4 架構，DiffusionGemma 使用離散擴散生成詞元。

> **Note:** 此筆記本需要記憶體超過 60GB 的 GPU，例如 NVIDIA G4 或 NVIDIA H100。


## 安裝 Python 套件

安裝執行 DiffusionGemma 模型與發送請求所需的 Hugging Face 函式庫。

In [ ]:
%%bash
# Install PyTorch & other libraries
pip install torch accelerate

# Install the transformers library
pip install "transformers>=5.11.0"


## 載入模型

使用 `transformers` 函式庫載入模型。

In [ ]:
from transformers import DiffusionGemmaForBlockDiffusion, AutoProcessor

MODEL_ID = "google/diffusiongemma-26B-A4B-it"

model = DiffusionGemmaForBlockDiffusion.from_pretrained(
    MODEL_ID, dtype="auto", device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)


使用提示範本來組織您的請求。提示範本讓您可以指定特定角色（例如 `user` 或 `model`）的輸入，是管理與 Gemma 模型多輪對話互動所需的格式。

In [ ]:
message = [
    {"role": "user", "content": "Why is the sky blue?"}
]

input_ids = processor.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
output = model.generate(**input_ids, max_new_tokens=512)

text = processor.decode(output[0], skip_special_tokens=False)
print(text[0])


使用 `TextDiffusionStreamer` 來查看擴散過程的中間狀態。

In [ ]:
from transformers import TextDiffusionStreamer

streamer = TextDiffusionStreamer(tokenizer=processor.tokenizer)

message = [
    {"role": "user", "content": "Why is the sky blue?"}
]

input_ids = processor.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
output = model.generate(**input_ids, max_new_tokens=512, streamer=streamer)

text = processor.decode(output[0], skip_special_tokens=False)
print("\n-- Final Output --")
print(text[0])


## 系統指令

使用 `system` 角色來提供系統層級的指令。

In [ ]:
message = [
    {"role": "system", "content": "Speak like a pirate."},
    {"role": "user", "content": "Why is the sky blue?"}
]

input_ids = processor.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
output = model.generate(**input_ids, max_new_tokens=512)

text = processor.decode(output[0], skip_special_tokens=False)
print(text[0])


## 使用思考功能的單次文字推論

要使用模型的思考能力生成回覆，請傳入 `enable_thinking=True`，處理器會在提示中插入正確的思考詞元，指示模型先思考再回覆。

In [ ]:
message = [
    {"role": "user", "content": "What is the water formula?"}
]

input_ids = processor.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt", enable_thinking=True).to(model.device)
output = model.generate(**input_ids, max_new_tokens=512)

text = processor.decode(output[0], skip_special_tokens=False)
print(text[0])


## 影像理解

以下範例示範如何提供一張影像並針對它提問。

In [ ]:
from PIL import Image
from IPython.display import display
import requests

img_url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/GoldenGate.png"
input_image = Image.open(requests.get(img_url, stream=True).raw)
display(input_image)

message = [
    {
        "role": "user", "content": [
            {"type": "image", "url": img_url},
            {"type": "text", "text": "What is shown in this image?"}
        ]
    }
]

input_ids = processor.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
output = model.generate(**input_ids, max_new_tokens=512)

text = processor.decode(output[0], skip_special_tokens=False)
print(text[0])


## 摘要

在本指南中，您學會了如何使用 DiffusionGemma 執行各種任務。